# SisaKulkas - Gabung Dataset dan Training v3

Notebook ini menggabungkan **dua dataset Roboflow** di Colab, lalu melatih model dari data gabungan.
(Fitur Merge Projects bawaan Roboflow hanya untuk proyek di workspace yang sama, sedangkan kedua dataset ini ada di workspace berbeda.)

**Urutan kerja**
1. Jalankan sel 1 sampai 4 (pengaturan, unduh, tampilkan kelas kedua dataset).
2. Lihat daftar kelas, lalu isi `RENAME` dan `DROP` di sel pemetaan agar nama kelas yang sama (misalnya `chicken` dan `Ayam`) disatukan.
3. Ubah `CONFIRMED = True`, lalu jalankan sel penggabungan.
4. Lanjutkan ke sel berikutnya (diagnosa, training, ekspor ONNX).

**Sebelum mulai:** Runtime > Change runtime type > **T4 GPU**. Simpan API key Roboflow yang **baru** di **Secrets** dengan nama `ROBOFLOW_API_KEY` (aktifkan *Notebook access*). Jangan menulis key di kode.

In [ ]:
!pip install -q roboflow ultralytics

## 1. Pengaturan

In [ ]:
# --- Dataset 1 (utama) ---
WORKSPACE1 = "veraluckyray282002-gmail-com"
PROJECT1   = "laststretch"
VERSION1   = 1

# --- Dataset 2 (yang digabung) ---
WORKSPACE2 = "penelitian-dikti-deteksi-objek-protein"
PROJECT2   = "deteksi-objek-protein-original-dataset-don-t-delete"
VERSION2   = 9

# --- Training ---
MODEL  = "yolov8s.pt"  # yolov8n.pt (tercepat) | yolov8s.pt (disarankan) | yolov8m.pt (lebih akurat, lebih berat)
IMGSZ  = 640           # 640 cepat. 800 membantu objek kecil, tapi deteksi di web jadi lebih lambat
EPOCHS = 100
BATCH  = 16            # jika muncul "CUDA out of memory", turunkan ke 8

## 2. Unduh kedua dataset

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

ds1 = rf.workspace(WORKSPACE1).project(PROJECT1).version(VERSION1).download("yolov8", location="/content/ds1")
try:
    ds2 = rf.workspace(WORKSPACE2).project(PROJECT2).version(VERSION2).download("yolov8", location="/content/ds2")
except Exception as e:
    raise SystemExit(
        "Dataset 2 tidak bisa diunduh dengan API key ini: " + str(e) + "\n"
        "Solusi: di Roboflow buka dataset 2 > Download Dataset > format YOLOv8 > download zip, "
        "unggah ke Colab, lalu ekstrak ke folder /content/ds2 (harus berisi data.yaml, train/, valid/).")
print("Dataset 1:", ds1.location, "| Dataset 2:", ds2.location)

## 3. Lihat kelas kedua dataset
Bandingkan nama kelas. Kelas yang sama tapi namanya beda (bahasa Inggris vs Indonesia, huruf besar/kecil) harus disatukan di langkah berikutnya.

In [ ]:
import os, glob, hashlib, shutil, yaml
from collections import Counter, defaultdict

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
SPLIT_DIRS = {"valid": ["valid", "val"], "test": ["test"], "train": ["train"]}

def _norm(s):
    return str(s).strip().lower()

def read_names(ds_dir):
    with open(os.path.join(ds_dir, "data.yaml")) as f:
        n = yaml.safe_load(f)["names"]
    return [n[i] for i in sorted(n)] if isinstance(n, dict) else list(n)

def merge_datasets(ds_dirs, tags, rename, drop, out):
    """Gabungkan dataset format YOLO. Kembalikan (nama_kelas, statistik, objek_per_kelas)."""
    ren = {_norm(k): v for k, v in rename.items()}
    drp = {_norm(x) for x in drop}

    final, final_ix, maps = [], {}, []
    for d in ds_dirs:
        m = {}
        for old_id, name in enumerate(read_names(d)):
            new_name = ren.get(_norm(name), name)
            if _norm(name) in drp or _norm(new_name) in drp:
                m[old_id] = None
                continue
            key = _norm(new_name)
            if key not in final_ix:
                final_ix[key] = len(final)
                final.append(new_name)
            m[old_id] = final_ix[key]
        maps.append(m)

    if os.path.exists(out):
        shutil.rmtree(out)
    seen = set()
    stats, per_class = Counter(), defaultdict(Counter)

    # valid dan test diproses lebih dulu: jika ada gambar kembar, salinan di train yang dibuang
    for split in ["valid", "test", "train"]:
        for d, tag, m in zip(ds_dirs, tags, maps):
            sdir = next((os.path.join(d, s) for s in SPLIT_DIRS[split]
                         if os.path.isdir(os.path.join(d, s, "images"))), None)
            if sdir is None:
                continue
            for img in sorted(glob.glob(os.path.join(sdir, "images", "*"))):
                if not img.lower().endswith(IMG_EXT):
                    continue
                with open(img, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                if h in seen:
                    stats["gambar kembar dibuang"] += 1
                    continue
                stem, ext = os.path.splitext(os.path.basename(img))
                lbl = os.path.join(sdir, "labels", stem + ".txt")
                lines = []
                if os.path.exists(lbl):
                    for ln in open(lbl).read().splitlines():
                        p = ln.split()
                        if len(p) >= 5:
                            lines.append(p)
                new_lines = []
                for p in lines:
                    nid = m.get(int(p[0]))
                    if nid is not None:
                        new_lines.append(" ".join([str(nid)] + p[1:]))
                if lines and not new_lines:
                    stats["gambar dilewati (semua objeknya dibuang)"] += 1
                    continue
                seen.add(h)
                os.makedirs(f"{out}/{split}/images", exist_ok=True)
                os.makedirs(f"{out}/{split}/labels", exist_ok=True)
                name = f"{tag}_{stem}"
                shutil.copy(img, f"{out}/{split}/images/{name}{ext}")
                with open(f"{out}/{split}/labels/{name}.txt", "w") as f:
                    f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
                stats[f"{split}: gambar"] += 1
                for ln in new_lines:
                    per_class[split][final[int(ln.split()[0])]] += 1

    cfg = {"path": out, "train": "train/images", "val": "valid/images", "names": final}
    if os.path.isdir(f"{out}/test/images"):
        cfg["test"] = "test/images"
    with open(f"{out}/data.yaml", "w") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True)
    return final, stats, per_class


import pandas as pd
n1, n2 = read_names("/content/ds1"), read_names("/content/ds2")

def count_instances(ds_dir, names):
    cnt = Counter()
    for lf in glob.glob(f"{ds_dir}/*/labels/*.txt"):
        for line in open(lf):
            p = line.split()
            if len(p) >= 5:
                cnt[int(p[0])] += 1
    return [cnt[i] for i in range(len(names))]

print("DATASET 1 (laststretch):")
display(pd.DataFrame({"kelas": n1, "jumlah_objek": count_instances("/content/ds1", n1)}))
print("DATASET 2 (protein):")
display(pd.DataFrame({"kelas": n2, "jumlah_objek": count_instances("/content/ds2", n2)}))
same = sorted({x.strip().lower() for x in n1} & {x.strip().lower() for x in n2})
print("Nama kelas yang sudah sama (tanpa membedakan huruf besar/kecil):", same)

## 4. Pemetaan kelas
Isi `RENAME` untuk menyatukan nama (format `"nama di dataset": "nama akhir"`), dan `DROP` untuk membuang kelas. Gunakan nama kelas persis seperti di tabel di atas. Gambar yang **semua** objeknya dibuang akan dilewati, tetapi gambar yang hanya sebagian objeknya dibuang tetap dipakai (objek yang dibuang menjadi latar belakang).

In [ ]:
RENAME = {
    # contoh (hapus tanda # dan sesuaikan dengan tabel di atas):
    # "chicken": "Ayam",
    # "egg": "Telur",
}
DROP = {"Image_9"}   # kelas yang dibuang beserta anotasinya

CONFIRMED = False    # ubah menjadi True setelah RENAME dan DROP diperiksa

## 5. Gabungkan
Valid dan test diproses lebih dulu, sehingga jika ada gambar yang persis sama di train, salinan di train yang dibuang. Ini mengurangi risiko hasil evaluasi terlalu bagus karena data bocor.

In [ ]:
if not CONFIRMED:
    raise SystemExit("Periksa RENAME dan DROP dulu, lalu ubah CONFIRMED = True.")

OUT = "/content/merged"
final_names, stats, per_class = merge_datasets(["/content/ds1", "/content/ds2"], ["a", "b"], RENAME, DROP, OUT)

print(f"Jumlah kelas akhir: {len(final_names)}")
print(final_names)
print()
for k, v in sorted(stats.items()):
    print(f"{k}: {v}")
tbl = pd.DataFrame({s: pd.Series(per_class[s]) for s in ["train", "valid", "test"]}).fillna(0).astype(int)
tbl["jumlah_total"] = tbl.sum(axis=1)
display(tbl.sort_values("jumlah_total"))

class _Merged:      # sel-sel berikutnya memakai dataset.location
    location = OUT
dataset = _Merged()

## 3. Rapikan data.yaml
Mencari folder validasi yang ada. Jika tidak ada, 15% data train dipisah sebagai validasi.

In [ ]:
import os, glob, random, shutil, yaml

loc = dataset.location
has = lambda s: os.path.isdir(f"{loc}/{s}/images")

VAL = next((s for s in ["valid", "val"] if has(s)), None)
if VAL is None:
    os.makedirs(f"{loc}/valid/images", exist_ok=True)
    os.makedirs(f"{loc}/valid/labels", exist_ok=True)
    imgs = sorted(glob.glob(f"{loc}/train/images/*"))
    random.seed(0); random.shuffle(imgs)
    for img in imgs[:max(1, int(0.15 * len(imgs)))]:
        base = os.path.splitext(os.path.basename(img))[0]
        shutil.move(img, f"{loc}/valid/images/")
        lbl = f"{loc}/train/labels/{base}.txt"
        if os.path.exists(lbl):
            shutil.move(lbl, f"{loc}/valid/labels/")
    VAL = "valid"

with open(f"{loc}/data.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = loc
cfg["train"] = "train/images"
cfg["val"] = f"{VAL}/images"
cfg.pop("test", None)
if has("test"):
    cfg["test"] = "test/images"
with open(f"{loc}/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f)

DATA_YAML = f"{loc}/data.yaml"
names = cfg["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else names
print("Validasi:", VAL, "| jumlah kelas:", len(names))

## 4. Periksa data
Kelas dengan data sedikit hampir pasti sering terlewat. Objek yang sangat kecil di foto juga sulit dideteksi.

In [ ]:
from collections import Counter
import pandas as pd

def count_split(split):
    cnt, small, total = Counter(), 0, 0
    for lf in glob.glob(f"{loc}/{split}/labels/*.txt"):
        for line in open(lf):
            p = line.split()
            if len(p) >= 5:
                cnt[int(p[0])] += 1
                total += 1
                if float(p[3]) * float(p[4]) < 0.01:   # kurang dari 1% luas gambar
                    small += 1
    return cnt, small, total

tr, small_tr, tot_tr = count_split("train")
va, _, _ = count_split(VAL)
df = pd.DataFrame({"kelas": names,
                   "objek_train": [tr[i] for i in range(len(names))],
                   "objek_valid": [va[i] for i in range(len(names))]})
df["catatan"] = df["objek_train"].apply(lambda n: "DATA SEDIKIT (<100)" if n < 100 else "")
display(df.sort_values("objek_train").reset_index(drop=True))
print(f"Gambar train: {len(glob.glob(loc + '/train/images/*'))}, objek kecil (<1% luas gambar): {small_tr}/{tot_tr} ({100*small_tr/max(tot_tr,1):.0f}%)")

## 5. (Opsional) Diagnosa model lama
Unggah `best.pt` lama ke Colab (ikon folder di kiri). Tabel urut dari **recall terendah**: itulah bahan yang paling sering terlewat.

In [ ]:
from ultralytics import YOLO

def per_class(metrics, title):
    rows = []
    for j, ci in enumerate(metrics.box.ap_class_index):
        rows.append({"kelas": metrics.names[int(ci)], "precision": metrics.box.p[j],
                     "recall": metrics.box.r[j], "AP50": metrics.box.ap50[j], "AP50-95": metrics.box.ap[j]})
    out = pd.DataFrame(rows).sort_values("recall").round(3).reset_index(drop=True)
    print(title)
    display(out)
    return out

OLD = "best.pt"
if os.path.exists(OLD):
    m_old = YOLO(OLD).val(data=DATA_YAML, imgsz=640, verbose=False)
    per_class(m_old, f"Model lama (mAP50={m_old.box.map50:.3f}, recall={m_old.box.mr:.3f})")
else:
    print("best.pt lama tidak ditemukan, bagian ini dilewati.")

## 6. Latih model baru
Perubahan utama: model lebih besar, 100 epoch dengan *early stopping* (berhenti jika 30 epoch tidak membaik), augmentasi lebih beragam, dan mosaic dimatikan di 15 epoch terakhir.

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,
    cos_lr=True,
    close_mosaic=15,
    mosaic=1.0,
    mixup=0.1,
    scale=0.6,
    degrees=5,
    translate=0.1,
    fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    project="sisakulkas",
    name="v2",
    seed=0,
)
BEST = str(model.trainer.best)
print("Model terbaik:", BEST)

## 7. Evaluasi model baru

In [ ]:
best = YOLO(BEST)
m_new = best.val(data=DATA_YAML, imgsz=IMGSZ, verbose=False)
print(f"mAP50={m_new.box.map50:.3f}  mAP50-95={m_new.box.map:.3f}  precision={m_new.box.mp:.3f}  recall={m_new.box.mr:.3f}")
per_class(m_new, "Model baru per kelas (urut dari recall terendah)")

In [ ]:
from IPython.display import Image as IPImage, display
cm = f"{model.trainer.save_dir}/confusion_matrix_normalized.png"
if os.path.exists(cm):
    display(IPImage(cm))

## 8. Lihat contoh prediksi (ambang 25%)

In [ ]:
val_imgs = sorted(glob.glob(f"{loc}/{VAL}/images/*"))[:6]
preds = best.predict(val_imgs, conf=0.25, imgsz=IMGSZ, save=True, project="sisakulkas", name="contoh", exist_ok=True, verbose=False)
for p in preds:
    display(IPImage(os.path.join(p.save_dir, os.path.basename(p.path))))

## 9. Ekspor untuk aplikasi web
Menghasilkan `best.onnx` dan `config.json` (ukuran input + nama kelas). Ganti kedua file di folder aplikasi web: `model/best.onnx` dan `config.json` di folder utama.

In [ ]:
import json
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
class_names = [best.names[i] for i in sorted(best.names)]
with open("config.json", "w", encoding="utf-8") as f:
    json.dump({"imgsz": IMGSZ, "names": class_names}, f, ensure_ascii=False)
shutil.copy(onnx_path, "best.onnx")

from google.colab import files
files.download("best.onnx")
files.download("config.json")
files.download(BEST)   # best.pt baru, simpan sebagai cadangan

## Jika masih ada objek yang terlewat
- **Lihat tabel recall per kelas.** Kelas dengan recall rendah biasanya datanya sedikit atau tampilannya mirip kelas lain (lihat confusion matrix).
- **Tambah foto asli** dari kulkas/dapur nyata untuk kelas yang lemah, beri label di Roboflow, lalu training ulang. Ini perbaikan paling efektif, karena data online sering berbeda dari foto kulkas yang berantakan.
- **Coba `IMGSZ = 800`** jika banyak objek kecil, lalu ekspor ulang (config.json otomatis ikut berubah). Deteksi di web jadi lebih lambat.
- **Coba `yolov8m.pt`** jika GPU dan waktu cukup.
- Di aplikasi web, turunkan **ambang keyakinan** di Pengaturan untuk menampilkan lebih banyak objek.